<a href="https://colab.research.google.com/github/eschain93/ban5600_800/blob/main/week4/Homework_4_Chainani_Emma.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Homework 4

##Google Authentication and OpenAI API

In [1]:
# Main setup for the notebook

# Google authentication
from google.colab import auth
auth.authenticate_user()
print("Successfully authenticated with Google.")

# Imports
import os
import textwrap
from datetime import datetime
from zoneinfo import ZoneInfo

import gspread
import gspread.exceptions
import pandas as pd

from google.auth import default
from google.colab import userdata
from openai import OpenAI

# Connect to Google Sheets
creds, _ = default()
gc = gspread.authorize(creds)
print("Successfully connected to Google Sheets.")

# Connect to OpenAI
os.environ["OPENAI_API_KEY"] = userdata.get("openaikey")
client = OpenAI()
print("Successfully connected to the OpenAI API.")

Successfully authenticated with Google.
Successfully connected to Google Sheets.
Successfully connected to the OpenAI API.


###Problem 1: Tracking Low Inventory and Reorder Needs



**Description**

Inventory shortages can delay customer orders, create lost sales, and frustrate staff. In many workplaces, someone still checks stock levels manually in spreadsheets and decides what needs to be reordered. This project asks students to automate that review by identifying items below their reorder threshold and creating a report that a manager could use to trigger replenishment.

**Objective**

Build a Python workflow that reads an inventory file, detects low-stock items, and creates a reorder alert report.


In [2]:
# Open the relevant Google Sheet
sheet_name = "Inventory Log"
sh = gc.open(sheet_name)
worksheet = sh.sheet1

# Step 6: Read the worksheet into a pandas DataFrame
inventory_df = pd.DataFrame(worksheet.get_all_records())

# Preview the Data
display(inventory_df.head())
print("Total rows loaded:", len(inventory_df))
print("Data Source: ",sh.url)

,Item_ID,Item_Name,Category,Current_Stock,Reorder_Level,Supplier
0,1001,Printer Paper,Office Supplies,12,20,Staples Co
1,1002,Blue Pens,Office Supplies,45,25,Staples Co
2,1003,Coffee Pods,Beverage,8,15,Coffee Direct
3,1004,Sticky Notes,Office Supplies,30,10,Office Depot
4,1005,Bottled Water,Beverage,5,18,Fresh Water Inc


Total rows loaded: 50
Data Source:  https://docs.google.com/spreadsheets/d/1yHv6nVis1sBTAUuxRGht2cxfCfZjsQj2r1v-LI0mBnM


In [3]:
# Convert the DataFrame into a list of dictionaries
inventory_records = inventory_df.to_dict(orient="records")

# Create empty lists and a counter
low_stock_items = []
critical_items = []
total_low_stock = 0

# Review each inventory item
for item in inventory_records:
    current_stock = item["Current_Stock"]
    reorder_level = item["Reorder_Level"]

    # Skip items that do not need reordering
    if current_stock > reorder_level:
        continue

    # Calculate the shortage amount
    shortage = reorder_level - current_stock
    item["Shortage"] = shortage

    # Add the item to the low-stock list
    low_stock_items.append(item)
    total_low_stock += 1

    # Flag critically low items
    if current_stock <= 5:
        critical_items.append(item)

print("Total low-stock items:", total_low_stock)
print("Total critical items:", len(critical_items))

Total low-stock items: 31
Total critical items: 11


In [4]:
# Sort low-stock items from most urgent to least urgent
low_stock_items.sort(key=lambda x: x["Shortage"], reverse=True)

# Convert the low-stock list into a DataFrame
low_stock_df = pd.DataFrame(low_stock_items)

# Add a new checkbox column
low_stock_df["Reordered?"] = False

# Import needed modules
from datetime import datetime
import gspread.exceptions
from zoneinfo import ZoneInfo

# Create a tab name with today's date
today = datetime.now(ZoneInfo("America/New_York")).strftime("%Y-%m-%d")
report_tab_name = f"Low Stock Report {today}"

print("Trying to write report to tab:", report_tab_name)

# Create the worksheet if it does not exist, otherwise open and clear it
try:
    report_worksheet = sh.worksheet(report_tab_name)
    report_worksheet.clear()
    print(f"Opened existing worksheet: {report_tab_name}")
except gspread.exceptions.WorksheetNotFound:
    report_worksheet = sh.add_worksheet(title=report_tab_name, rows=100, cols=20)
    print(f"Created new worksheet: {report_tab_name}")

# Prepare data for Google Sheets
report_data = [low_stock_df.columns.tolist()] + low_stock_df.values.tolist()

# Write the report starting in cell A1
report_worksheet.update(values=report_data, range_name="A1")

# Find worksheet ID and checkbox column index
sheet_id = report_worksheet.id
num_rows = len(low_stock_df) + 1   # +1 for header row
num_cols = len(low_stock_df.columns)

# "Reordered?" is the last column
checkbox_col_index = num_cols - 1

# Apply checkbox validation to all data rows in that column
sh.batch_update({
    "requests": [
        {
            "setDataValidation": {
                "range": {
                    "sheetId": sheet_id,
                    "startRowIndex": 1,   # skip header row
                    "endRowIndex": num_rows,
                    "startColumnIndex": checkbox_col_index,
                    "endColumnIndex": checkbox_col_index + 1
                },
                "rule": {
                    "condition": {
                        "type": "BOOLEAN"
                    },
                    "showCustomUi": True,
                    "strict": True
                }
            }
        }
    ]
})

print("Checkboxes added to the 'Reordered?' column.")

# Apply conditional formatting to highlight critical low-stock rows in red
sheet_id = report_worksheet.id
num_rows = len(low_stock_df) + 1   # include header row
num_cols = len(low_stock_df.columns)

sh.batch_update({
    "requests": [
        {
            "addConditionalFormatRule": {
                "rule": {
                    "ranges": [
                        {
                            "sheetId": sheet_id,
                            "startRowIndex": 1,          # skip header
                            "endRowIndex": num_rows,
                            "startColumnIndex": 0,
                            "endColumnIndex": num_cols
                        }
                    ],
                    "booleanRule": {
                        "condition": {
                            "type": "CUSTOM_FORMULA",
                            "values": [
                                {"userEnteredValue": "=$D2<=5"}
                            ]
                        },
                        "format": {
                            "backgroundColor": {
                                "red": 1,
                                "green": 0.8,
                                "blue": 0.8
                            }
                        }
                    }
                },
                "index": 0
            }
        }
    ]
})

# Create Critical Alerts Preview
print("CRITICAL ALERT PREVIEW")
print("-" * 40)

index = 0
while index < len(critical_items):
    item = critical_items[index]
    print("Item ID:", item["Item_ID"], "| Item Name:", item["Item_Name"])
    index += 1

    if index == 3:
        break

print("Low-stock report written to worksheet:", report_tab_name, sh.url)
print("-" * 40)
display(low_stock_df)

Trying to write report to tab: Low Stock Report 2026-04-21
Created new worksheet: Low Stock Report 2026-04-21
Checkboxes added to the 'Reordered?' column.
CRITICAL ALERT PREVIEW
----------------------------------------
Item ID: 1005 | Item Name: Bottled Water
Item ID: 1007 | Item Name: Toner Cartridge
Item ID: 1015 | Item Name: USB Cables
Low-stock report written to worksheet: Low Stock Report 2026-04-21 https://docs.google.com/spreadsheets/d/1yHv6nVis1sBTAUuxRGht2cxfCfZjsQj2r1v-LI0mBnM
----------------------------------------


,Item_ID,Item_Name,Category,Current_Stock,Reorder_Level,Supplier,Shortage,Reordered?
0,1005,Bottled Water,Beverage,5,18,Fresh Water Inc,13,False
1,1011,Shipping Boxes,Packaging,14,25,PackRight,11,False
2,1015,USB Cables,Electronics,4,14,TechSource,10,False
3,1023,Dry Erase Markers,Office Supplies,2,12,Staples Co,10,False
4,1050,Coffee Stirrers,Beverage,7,17,Breakroom Supply,10,False
5,1013,Bubble Wrap,Packaging,9,18,PackRight,9,False
6,1017,Paper Towels,Maintenance,7,16,Clean Supply LLC,9,False
7,1001,Printer Paper,Office Supplies,12,20,Staples Co,8,False
8,1027,Napkins,Beverage,6,14,Breakroom Supply,8,False
9,1040,Creamer Cups,Beverage,2,10,Coffee Direct,8,False


In [5]:
import textwrap

# Initialize client
client = OpenAI()

# Build a compact summary for the AI to interpret
top_shortages = low_stock_df[["Item_ID", "Item_Name", "Category", "Current_Stock", "Reorder_Level", "Shortage"]].head(5).to_dict(orient="records")
category_counts = low_stock_df["Category"].value_counts().to_dict()

response = client.chat.completions.create(
    model="gpt-4o",
    messages=[
        {
            "role": "user",
            "content": f"""
Interpret this low-inventory report.

Total low-stock items: {total_low_stock}
Total critical items: {len(critical_items)}
Low-stock items by category: {category_counts}
Top 5 shortages: {top_shortages}

Write a short business explanation in 3 to 5 sentences.
Keep it specific to the results shown.
Explain how this automation supports inventory control.
Do not use bullet points.
"""
        }
    ]
)

business_explanation = response.choices[0].message.content
print("\nBusiness Explanation:\n")
for paragraph in business_explanation.split("\n"):
    if paragraph.strip():
        print(textwrap.fill(paragraph, width=80))
        print()


Business Explanation:

The low-inventory report indicates that there are currently 31 products with low
stock, of which 11 are deemed critical. The categories most affected include
Office Supplies and Beverages, with 11 and 8 low-stock items respectively.
Notably, Bottled Water and Coffee Stirrers are facing significant shortages of
13 and 10 units, while Dry Erase Markers and USB Cables also stand out in terms
of depleted inventory. This automation is vital for inventory control as it
provides timely insights into which products require immediate restocking,
allowing the business to prioritize and allocate resources efficiently to
prevent disruptions in operations.





---



###Problem 2: Organizing Customer or Client Email Responses

**Description**

In many customer-facing jobs, employees answer the same kinds of messages repeatedly, such as confirmations, reminders, shipping updates, or product questions. Writing similar replies again and again wastes time and can create inconsistent wording. This assignment asks students to generate standardized, personalized response drafts with Python.

**Objective**

Create a Python notebook that generates personalized email response drafts based on request type and customer details.


In [6]:
# Open the spreadsheet
sheet_name = "Automated Customer Email Responses"
sh = gc.open(sheet_name)

# Open the first worksheet
worksheet = sh.sheet1

# Read the worksheet into a pandas DataFrame
customer_df = pd.DataFrame(worksheet.get_all_records())

# Preview the data
display(customer_df.head())
print("Total rows loaded:", len(customer_df))
print("Data Source:", sh.url)

,Customer_Name,Email,Request_Type,Account_Status,Reference_Date
0,Alex Morgan,alex.morgan@email.com,Appointment Confirmation,Active,2026-04-10
1,Brianna Lee,brianna.lee@email.com,Payment Reminder,Past Due,2026-04-12
2,Carlos Rivera,carlos.rivera@email.com,Lab Results Follow-Up,Active,2026-04-15
3,Dana Patel,dana.patel@email.com,Prescription Refill Question,Active,2026-04-09
4,Ethan Brooks,ethan.brooks@email.com,Appointment Confirmation,Active,2026-04-11


Total rows loaded: 50
Data Source: https://docs.google.com/spreadsheets/d/1PaUJdzqAiuSQgqE2Fiu8AeCtAmnIPZc0xNo6oszczN0


In [7]:
# Convert the DataFrame into a list of dictionaries
customer_records = customer_df.to_dict(orient="records")

# Create an empty list to store generated email drafts
email_drafts = []
missing_email_customers = []

print("Total customer records ready for processing:", len(customer_records))

Total customer records ready for processing: 50


In [8]:
# Review each customer record
for customer in customer_records:
    customer_name = customer["Customer_Name"]
    email = str(customer["Email"]).strip()
    request_type = customer["Request_Type"]
    account_status = customer["Account_Status"]
    reference_date = customer["Reference_Date"]

    # Skip records with missing email addresses
    if email == "":
        missing_email_customers.append(customer_name)
        continue

    # Choose the email template based on request type
    if request_type == "Appointment Confirmation":
        message = (
            f"Hello {customer_name}, this is a confirmation of your appointment scheduled for "
            f"{reference_date}. Please contact our office if you need to reschedule."
        )

    elif request_type == "Payment Reminder":
        if account_status == "Past Due":
            message = (
                f"Hello {customer_name}, this is a reminder that your account has a past due balance. "
                f"Please submit payment by {reference_date} or contact our office if you have any questions."
            )
        else:
            message = (
                f"Hello {customer_name}, this is a friendly reminder about your upcoming payment due on "
                f"{reference_date}."
            )

    elif request_type == "Lab Results Follow-Up":
        message = (
            f"Hello {customer_name}, this is a follow-up regarding your lab results. "
            f"Please check your patient portal or contact our office on or after {reference_date} for an update."
        )

    elif request_type == "Prescription Refill Question":
        message = (
            f"Hello {customer_name}, we received your prescription refill question. "
            f"Our office is reviewing your request and will follow up by {reference_date}."
        )

    else:
        message = (
            f"Hello {customer_name}, thank you for contacting our office. "
            f"We will follow up with you as soon as possible."
        )

    # Store the generated draft
    email_drafts.append({
        "Customer_Name": customer_name,
        "Email": email,
        "Request_Type": request_type,
        "Account_Status": account_status,
        "Reference_Date": reference_date,
        "Draft_Message": message
    })

print("Total email drafts created:", len(email_drafts))
print("Total customers missing email addresses:", len(missing_email_customers))

Total email drafts created: 46
Total customers missing email addresses: 4


In [9]:
# Convert the list of generated drafts into a DataFrame
email_drafts_df = pd.DataFrame(email_drafts)

# Preview the generated drafts
display(email_drafts_df.head())

print("Total email drafts created:", len(email_drafts_df))

print("EMAIL DRAFT PREVIEW")
print("-" * 40)

index = 0
while index < len(email_drafts_df):
    row = email_drafts_df.iloc[index]
    print(
        "Customer:", row["Customer_Name"],
        "| Request Type:", row["Request_Type"]
    )
    index += 1

    if index == 5:
        print("Preview stopped after 5 drafts to keep output concise.")
        break

,Customer_Name,Email,Request_Type,Account_Status,Reference_Date,Draft_Message
0,Alex Morgan,alex.morgan@email.com,Appointment Confirmation,Active,2026-04-10,"Hello Alex Morgan, this is a confirmation of y..."
1,Brianna Lee,brianna.lee@email.com,Payment Reminder,Past Due,2026-04-12,"Hello Brianna Lee, this is a reminder that you..."
2,Carlos Rivera,carlos.rivera@email.com,Lab Results Follow-Up,Active,2026-04-15,"Hello Carlos Rivera, this is a follow-up regar..."
3,Dana Patel,dana.patel@email.com,Prescription Refill Question,Active,2026-04-09,"Hello Dana Patel, we received your prescriptio..."
4,Ethan Brooks,ethan.brooks@email.com,Appointment Confirmation,Active,2026-04-11,"Hello Ethan Brooks, this is a confirmation of ..."


Total email drafts created: 46
EMAIL DRAFT PREVIEW
----------------------------------------
Customer: Alex Morgan | Request Type: Appointment Confirmation
Customer: Brianna Lee | Request Type: Payment Reminder
Customer: Carlos Rivera | Request Type: Lab Results Follow-Up
Customer: Dana Patel | Request Type: Prescription Refill Question
Customer: Ethan Brooks | Request Type: Appointment Confirmation
Preview stopped after 5 drafts to keep output concise.


In [10]:
# Create a tab name with today's date
from datetime import datetime
import gspread.exceptions
from zoneinfo import ZoneInfo

today = datetime.now(ZoneInfo("America/New_York")).strftime("%Y-%m-%d")
draft_tab_name = f"Email Drafts {today}"

print("Trying to write drafts to tab:", draft_tab_name)

# Create the worksheet if it does not exist, otherwise open and clear it
try:
    draft_worksheet = sh.worksheet(draft_tab_name)
    draft_worksheet.clear()
    print(f"Opened existing worksheet: {draft_tab_name}")
except gspread.exceptions.WorksheetNotFound:
    draft_worksheet = sh.add_worksheet(title=draft_tab_name, rows=200, cols=20)
    print(f"Created new worksheet: {draft_tab_name}")

# Prepare data for Google Sheets
draft_data = [email_drafts_df.columns.tolist()] + email_drafts_df.values.tolist()

# Write the drafts starting in cell A1
draft_worksheet.update(values=draft_data, range_name="A1")

print("Email drafts written to worksheet:", draft_tab_name)
print("Spreadsheet URL:", sh.url)

display(email_drafts_df.head())

Trying to write drafts to tab: Email Drafts 2026-04-21
Created new worksheet: Email Drafts 2026-04-21
Email drafts written to worksheet: Email Drafts 2026-04-21
Spreadsheet URL: https://docs.google.com/spreadsheets/d/1PaUJdzqAiuSQgqE2Fiu8AeCtAmnIPZc0xNo6oszczN0


,Customer_Name,Email,Request_Type,Account_Status,Reference_Date,Draft_Message
0,Alex Morgan,alex.morgan@email.com,Appointment Confirmation,Active,2026-04-10,"Hello Alex Morgan, this is a confirmation of y..."
1,Brianna Lee,brianna.lee@email.com,Payment Reminder,Past Due,2026-04-12,"Hello Brianna Lee, this is a reminder that you..."
2,Carlos Rivera,carlos.rivera@email.com,Lab Results Follow-Up,Active,2026-04-15,"Hello Carlos Rivera, this is a follow-up regar..."
3,Dana Patel,dana.patel@email.com,Prescription Refill Question,Active,2026-04-09,"Hello Dana Patel, we received your prescriptio..."
4,Ethan Brooks,ethan.brooks@email.com,Appointment Confirmation,Active,2026-04-11,"Hello Ethan Brooks, this is a confirmation of ..."


In [11]:
# Summary values for AI interpretation
request_type_counts = email_drafts_df["Request_Type"].value_counts().to_dict()
total_drafts = len(email_drafts_df)
total_missing_emails = len(missing_email_customers)

print("Total email drafts created:", total_drafts)
print("Total customers missing email addresses:", total_missing_emails)
print("Draft counts by request type:", request_type_counts)

Total email drafts created: 46
Total customers missing email addresses: 4
Draft counts by request type: {'Appointment Confirmation': 13, 'Payment Reminder': 12, 'Prescription Refill Question': 12, 'Lab Results Follow-Up': 9}


In [12]:
import textwrap

# Ask AI for a short business explanation
response = client.chat.completions.create(
    model="gpt-4o",
    messages=[
        {
            "role": "user",
            "content": f"""
Interpret this customer email automation summary.

Total email drafts created: {total_drafts}
Total customers missing email addresses: {total_missing_emails}
Draft counts by request type: {request_type_counts}

Write a short business explanation in 3 to 5 sentences.
Explain how template-based email automation saves time and supports consistency.
Do not use bullet points.
"""
        }
    ]
)

business_explanation = response.choices[0].message.content

print("\nBusiness Explanation:\n")
print(textwrap.fill(business_explanation, width=90))


Business Explanation:

The recent email automation summary reveals the efficiency and effectiveness of using
template-based systems in customer communication. With a total of 46 email drafts created,
the automation not only streamlines the process of responding to common customer requests,
such as appointment confirmations and payment reminders, but it also saves significant
time that would otherwise be spent on drafting individual emails. Although four customers
are missing email addresses, the predefined templates ensure that all other communications
are consistent and professional. By utilizing these templates, businesses can maintain a
high standard of customer service while also optimizing their workflow and reducing the
potential for human error.




---



###Problem 3: Validating Application or Form Data Before Processing

**Description**

Organizations in healthcare, insurance, education, government, and banking often receive forms that are missing important information or contain invalid entries. Manually checking every record is time-consuming and repetitive. This assignment asks students to automate basic validation checks so incomplete applications can be flagged before staff spend time processing them.

**Objective**

Build a Python notebook that validates application records and identifies incomplete or invalid submissions.


In [13]:
# Open the spreadsheet
sheet_name = "Application Validation Log"
sh = gc.open(sheet_name)

# Open the first worksheet
worksheet = sh.sheet1

# Read the worksheet into a pandas DataFrame
application_df = pd.DataFrame(worksheet.get_all_records())

# Preview the data
display(application_df.head())
print("Total rows loaded:", len(application_df))
print("Data Source:", sh.url)

,Applicant_Name,Email,Phone,Date_of_Birth,Application_Status
0,Monica Alvarez,monica.alvarez@email.com,9085551201,1993-04-18,Pending
1,Trevor Nash,trevor.nash@email.com,7325553478,1987-09-22,Approved
2,Janelle Brooks,janelle.brooks@email.com,6095559087,1999-01-11,Pending
3,Marcus Holloway,marcus.holloway@email.com,8565557721,1995-06-30,Review
4,Selena Vaughn,selena.vaughn@email.com,8485556644,1991-12-05,Pending


Total rows loaded: 50
Data Source: https://docs.google.com/spreadsheets/d/1sIDEsPcDHQu8nZMAyt4JcyUIijaHICgYns8ypZz9moY


In [14]:
# Convert the DataFrame into a list of dictionaries
application_records = application_df.to_dict(orient="records")

# Create empty lists for results
flagged_records = []
valid_records = []

print("Total application records ready for validation:", len(application_records))

Total application records ready for validation: 50


In [15]:
# Review each application record
for application in application_records:
    applicant_name = application["Applicant_Name"]
    email = str(application["Email"]).strip()
    phone = str(application["Phone"]).strip()
    dob = str(application["Date_of_Birth"]).strip()
    status = application["Application_Status"]

    validation_issues = []

    # Rule 1: Check for missing or invalid email
    if email == "":
        validation_issues.append("Missing email")
    elif "@" not in email or "." not in email:
        validation_issues.append("Invalid email format")

    # Rule 2: Check for invalid phone number
    phone_digits = "".join(filter(str.isdigit, phone))
    if len(phone_digits) != 10:
        validation_issues.append("Invalid phone number")

    # Rule 3: Check for missing or unrealistic date of birth
    if dob == "":
        validation_issues.append("Missing date of birth")
    else:
        try:
            dob_year = int(dob[:4])
            if dob_year < 1926 or dob_year > 2008:
                validation_issues.append("Unrealistic date of birth")
        except:
            validation_issues.append("Invalid date of birth format")

    # Save flagged or valid records
    if len(validation_issues) > 0:
        flagged_records.append({
            "Applicant_Name": applicant_name,
            "Email": email,
            "Phone": phone,
            "Date_of_Birth": dob,
            "Application_Status": status,
            "Validation_Issue": ", ".join(validation_issues)
        })
        continue

    valid_records.append(application)

print("Total flagged applications:", len(flagged_records))
print("Total valid applications:", len(valid_records))

Total flagged applications: 10
Total valid applications: 40


In [16]:
flagged_df = pd.DataFrame(flagged_records)

total_flagged = len(flagged_records)
total_valid = len(valid_records)
issue_counts = flagged_df["Validation_Issue"].value_counts().to_dict()

display(flagged_df.head())
print("Total flagged applications:", len(flagged_df))

print("FLAGGED APPLICATION PREVIEW")
print("-" * 40)

index = 0
while index < len(flagged_records):
    record = flagged_records[index]
    print("Applicant:", record["Applicant_Name"], "| Issue:", record["Validation_Issue"])
    index += 1

    if index == 5:
        print("The remaining flagged applicants can be located in the spreadsheet:", sh.url)
        break

,Applicant_Name,Email,Phone,Date_of_Birth,Application_Status,Validation_Issue
0,Derrick Sloan,,7325558812,1994-03-27,Pending,Missing email
1,Jasper Quinn,jasper.quinn@email.com,abc5551122,1985-05-16,Pending,Invalid phone number
2,Bennett Cross,bennett.cross@email.com,6095556677,,Pending,Missing date of birth
3,Wyatt Henson,wyatt.henson@email.com,9085555566,2011-04-10,Pending,Unrealistic date of birth
4,Holden Pierce,,6095552288,1995-07-07,Pending,Missing email


Total flagged applications: 10
FLAGGED APPLICATION PREVIEW
----------------------------------------
Applicant: Derrick Sloan | Issue: Missing email
Applicant: Jasper Quinn | Issue: Invalid phone number
Applicant: Bennett Cross | Issue: Missing date of birth
Applicant: Wyatt Henson | Issue: Unrealistic date of birth
Applicant: Holden Pierce | Issue: Missing email
The remaining flagged applicants can be located in the spreadsheet: https://docs.google.com/spreadsheets/d/1sIDEsPcDHQu8nZMAyt4JcyUIijaHICgYns8ypZz9moY


In [17]:
from datetime import datetime
from zoneinfo import ZoneInfo
import gspread.exceptions

# Create a tab name with today's date using Eastern time
today = datetime.now(ZoneInfo("America/New_York")).strftime("%Y-%m-%d")
flagged_tab_name = f"Flagged Applications {today}"

print("Trying to write flagged records to tab:", flagged_tab_name)

# Create the worksheet if it does not exist, otherwise open and clear it
try:
    flagged_worksheet = sh.worksheet(flagged_tab_name)
    flagged_worksheet.clear()
    print(f"Opened existing worksheet: {flagged_tab_name}")
except gspread.exceptions.WorksheetNotFound:
    flagged_worksheet = sh.add_worksheet(title=flagged_tab_name, rows=200, cols=20)
    print(f"Created new worksheet: {flagged_tab_name}")

# Prepare data for Google Sheets
flagged_data = [flagged_df.columns.tolist()] + flagged_df.values.tolist()

# Write the flagged records starting in cell A1
flagged_worksheet.update(values=flagged_data, range_name="A1")

print("Flagged application report written to worksheet:", flagged_tab_name)
print("Spreadsheet URL:", sh.url)

display(flagged_df.head())

Trying to write flagged records to tab: Flagged Applications 2026-04-21
Created new worksheet: Flagged Applications 2026-04-21
Flagged application report written to worksheet: Flagged Applications 2026-04-21
Spreadsheet URL: https://docs.google.com/spreadsheets/d/1sIDEsPcDHQu8nZMAyt4JcyUIijaHICgYns8ypZz9moY


,Applicant_Name,Email,Phone,Date_of_Birth,Application_Status,Validation_Issue
0,Derrick Sloan,,7325558812,1994-03-27,Pending,Missing email
1,Jasper Quinn,jasper.quinn@email.com,abc5551122,1985-05-16,Pending,Invalid phone number
2,Bennett Cross,bennett.cross@email.com,6095556677,,Pending,Missing date of birth
3,Wyatt Henson,wyatt.henson@email.com,9085555566,2011-04-10,Pending,Unrealistic date of birth
4,Holden Pierce,,6095552288,1995-07-07,Pending,Missing email


In [18]:
import textwrap

# Analyze the data using AI

response = client.chat.completions.create(
    model="gpt-4o",
    messages=[
        {
            "role": "user",
            "content": f"""
Interpret this application validation summary.

Total flagged applications: {total_flagged}
Total valid applications: {total_valid}
Validation issue counts: {issue_counts}

Write a short business explanation in 3 to 5 sentences.
Explain how this validation process helps reduce manual review time and improve data quality.
Do not use bullet points.
"""
        }
    ]
)

business_explanation = response.choices[0].message.content

print("\nBusiness Explanation:\n")
print(textwrap.fill(business_explanation, width=90))


Business Explanation:

The application validation summary indicates that out of 50 applications processed, 10
were flagged due to errors such as missing email addresses, invalid phone numbers,
unrealistic dates of birth, and one missing date of birth. This automated validation
process significantly reduces the time and effort involved in manual reviews by
immediately identifying applications with these common issues. By ensuring that only
complete and correctly filled applications proceed, the validation process enhances data
quality, leading to more accurate and reliable information for decision-making.
Additionally, addressing these issues upfront allows for quicker correction and
resubmission, streamlining the application workflow.




---



###Problem 4: Tracking Task Deadlines and Follow-Ups

**Description**

Professionals often manage many deadlines at once, including project milestones, compliance tasks, and client follow-ups. When everything is tracked manually, it becomes easy to miss an urgent due date. This assignment gives students a practical workflow for reading a task list, identifying what is overdue or coming soon, and creating a reminder summary.

**Objective**

Develop a Python notebook that reads task data, identifies upcoming and overdue work, and generates a reminder report.


In [19]:
# Open the spreadsheet
sheet_name = "Task Deadline Tracker"
sh = gc.open(sheet_name)

# Open the first worksheet
worksheet = sh.sheet1

# Read the worksheet into a pandas DataFrame
task_df = pd.DataFrame(worksheet.get_all_records())

# Preview the data
display(task_df.head())
print("Total rows loaded:", len(task_df))
print("Data Source:", sh.url)

,Task_ID,Task_Name,Assigned_To,Due_Date,Priority,Status
0,T001,Submit insurance renewal form,Emma,2026-04-26,High,Open
1,T002,Prepare weekly project update,Daniel,2026-04-27,Medium,In Progress
2,T003,Review vendor contract,Sophia,2026-04-01,High,Completed
3,T004,Follow up with client on invoice,Marcus,2026-04-28,High,Open
4,T005,Update onboarding checklist,Alyssa,2026-04-21,Low,Open


Total rows loaded: 50
Data Source: https://docs.google.com/spreadsheets/d/1_7rA0Is5LQN_t7drrMSMj19tSWSldVxVFvZTUdYUKqc


In [20]:
# Convert the DataFrame into a list of dictionaries
task_records = task_df.to_dict(orient="records")

# Create empty lists for results
overdue_tasks = []
upcoming_tasks = []

print("Total task records ready for processing:", len(task_records))

Total task records ready for processing: 50


In [21]:
from datetime import datetime
from zoneinfo import ZoneInfo

# Set today's date using Eastern time
today = datetime.now(ZoneInfo("America/New_York")).date()

# Review each task record
for task in task_records:
    task_id = task["Task_ID"]
    task_name = task["Task_Name"]
    assigned_to = task["Assigned_To"]
    due_date = datetime.strptime(task["Due_Date"], "%Y-%m-%d").date()
    priority = task["Priority"]
    status = task["Status"]

    # Skip completed tasks
    if status == "Completed":
        continue

    days_until_due = (due_date - today).days

    # Overdue tasks
    if days_until_due < 0:
        overdue_tasks.append({
            "Task_ID": task_id,
            "Task_Name": task_name,
            "Assigned_To": assigned_to,
            "Due_Date": due_date.strftime("%Y-%m-%d"),
            "Priority": priority,
            "Status": status,
            "Days_Overdue": abs(days_until_due),
            "Deadline_Category": "Overdue"
        })

    # Tasks due within the next 7 days
    elif days_until_due <= 7:
        upcoming_tasks.append({
            "Task_ID": task_id,
            "Task_Name": task_name,
            "Assigned_To": assigned_to,
            "Due_Date": due_date.strftime("%Y-%m-%d"),
            "Priority": priority,
            "Status": status,
            "Days_Until_Due": days_until_due,
            "Deadline_Category": "Upcoming"
        })

print("Total overdue tasks:", len(overdue_tasks))
print("Total upcoming tasks:", len(upcoming_tasks))

Total overdue tasks: 10
Total upcoming tasks: 17


In [22]:
# Convert the task lists into DataFrames
overdue_df = pd.DataFrame(overdue_tasks)
upcoming_df = pd.DataFrame(upcoming_tasks)

# Combine both into one reminder report
reminder_df = pd.concat([overdue_df, upcoming_df], ignore_index=True)

# Preview the report
display(reminder_df.head())

print("Total overdue tasks:", len(overdue_df))
print("Total upcoming tasks:", len(upcoming_df))
print("Total tasks in reminder report:", len(reminder_df))

,Task_ID,Task_Name,Assigned_To,Due_Date,Priority,Status,Days_Overdue,Deadline_Category,Days_Until_Due
0,T012,Check software license renewal,Caleb,2026-04-20,High,Open,1.0,Overdue,NaN
1,T027,Prepare client follow-up list,Brody,2026-04-20,High,In Progress,1.0,Overdue,NaN
2,T036,Review support queue backlog,Bianca,2026-04-17,Medium,Open,4.0,Overdue,NaN
3,T037,Send updated timeline to client,Logan,2026-04-18,High,In Progress,3.0,Overdue,NaN
4,T038,Reconcile inventory adjustment,Marisol,2026-04-09,Medium,Open,12.0,Overdue,NaN


Total overdue tasks: 10
Total upcoming tasks: 17
Total tasks in reminder report: 27


In [23]:
# Sort reminder report by longest overdue first
reminder_df = reminder_df.sort_values(
    by="Days_Overdue",
    ascending=False,
    na_position="last"
).reset_index(drop=True)

display(reminder_df.head())

print("TASK REMINDER PREVIEW")
print("Overdue tasks in descending order")
print("-" * 40)

index = 0
while index < len(reminder_df):
    row = reminder_df.iloc[index]

    if row["Deadline_Category"] == "Overdue":
        category_text = f"Overdue +{int(row['Days_Overdue'])} days"
    else:
        category_text = f"Upcoming in {int(row['Days_Until_Due'])} days"

    print(
        "Task ID:", row["Task_ID"],
        "| Task Name:", row["Task_Name"],
        "| Category:", category_text
    )
    index += 1

    if index == 5:
        print("Preview stopped after 5 tasks to keep output concise.")
        break

,Task_ID,Task_Name,Assigned_To,Due_Date,Priority,Status,Days_Overdue,Deadline_Category,Days_Until_Due
0,T038,Reconcile inventory adjustment,Marisol,2026-04-09,Medium,Open,12.0,Overdue,NaN
1,T046,Approve reimbursement exception,Carter,2026-04-10,High,Open,11.0,Overdue,NaN
2,T042,Review monthly KPI report,Ruben,2026-04-12,High,In Progress,9.0,Overdue,NaN
3,T050,Send final reminder notice,Reid,2026-04-14,Medium,Open,7.0,Overdue,NaN
4,T044,Update project roadmap,Damien,2026-04-16,High,Open,5.0,Overdue,NaN


TASK REMINDER PREVIEW
Overdue tasks in descending order
----------------------------------------
Task ID: T038 | Task Name: Reconcile inventory adjustment | Category: Overdue +12 days
Task ID: T046 | Task Name: Approve reimbursement exception | Category: Overdue +11 days
Task ID: T042 | Task Name: Review monthly KPI report | Category: Overdue +9 days
Task ID: T050 | Task Name: Send final reminder notice | Category: Overdue +7 days
Task ID: T044 | Task Name: Update project roadmap | Category: Overdue +5 days
Preview stopped after 5 tasks to keep output concise.


In [24]:
from datetime import datetime
from zoneinfo import ZoneInfo
import gspread.exceptions

# Create a tab name with today's date using Eastern time
today = datetime.now(ZoneInfo("America/New_York")).strftime("%Y-%m-%d")
report_tab_name = f"Task Reminder Report {today}"

print("Trying to write reminder report to tab:", report_tab_name)

# Create the worksheet if it does not exist, otherwise open and clear it
try:
    report_worksheet = sh.worksheet(report_tab_name)
    report_worksheet.clear()
    print(f"Opened existing worksheet: {report_tab_name}")
except gspread.exceptions.WorksheetNotFound:
    report_worksheet = sh.add_worksheet(title=report_tab_name, rows=200, cols=20)
    print(f"Created new worksheet: {report_tab_name}")

# Replace NaN values with blanks before writing to Google Sheets
reminder_df = reminder_df.fillna("")

# Prepare data for Google Sheets
report_data = [reminder_df.columns.tolist()] + reminder_df.values.tolist()

# Write the reminder report starting in cell A1
report_worksheet.update(values=report_data, range_name="A1")

print("Task reminder report written to worksheet:", report_tab_name)
print("Spreadsheet URL:", sh.url)

display(reminder_df.head())

Trying to write reminder report to tab: Task Reminder Report 2026-04-21
Created new worksheet: Task Reminder Report 2026-04-21
Task reminder report written to worksheet: Task Reminder Report 2026-04-21
Spreadsheet URL: https://docs.google.com/spreadsheets/d/1_7rA0Is5LQN_t7drrMSMj19tSWSldVxVFvZTUdYUKqc


,Task_ID,Task_Name,Assigned_To,Due_Date,Priority,Status,Days_Overdue,Deadline_Category,Days_Until_Due
0,T038,Reconcile inventory adjustment,Marisol,2026-04-09,Medium,Open,12.0,Overdue,
1,T046,Approve reimbursement exception,Carter,2026-04-10,High,Open,11.0,Overdue,
2,T042,Review monthly KPI report,Ruben,2026-04-12,High,In Progress,9.0,Overdue,
3,T050,Send final reminder notice,Reid,2026-04-14,Medium,Open,7.0,Overdue,
4,T044,Update project roadmap,Damien,2026-04-16,High,Open,5.0,Overdue,


In [26]:
priority_counts = reminder_df["Priority"].value_counts().to_dict()
total_overdue = len(overdue_df)
total_upcoming = len(upcoming_df)

print("Total overdue tasks:", total_overdue)
print("Total upcoming tasks:", total_upcoming)
print("Task counts by priority:", priority_counts)

Total overdue tasks: 10
Total upcoming tasks: 17
Task counts by priority: {'High': 13, 'Medium': 9, 'Low': 5}


In [27]:
import textwrap

# Analyze the data using AI

response = client.chat.completions.create(
    model="gpt-4o",
    messages=[
        {
            "role": "user",
            "content": f"""
Interpret this task reminder summary.

Total overdue tasks: {total_overdue}
Total upcoming tasks: {total_upcoming}
Task counts by priority: {priority_counts}

Write a short business explanation in 3 to 5 sentences.
Explain how this automation supports project coordination and deadline management.
Do not use bullet points.
"""
        }
    ]
)

business_explanation = response.choices[0].message.content

print("\nBusiness Explanation:\n")
print(textwrap.fill(business_explanation, width=90))


Business Explanation:

This task reminder summary indicates a total of 10 overdue tasks and 17 upcoming tasks,
highlighting the current workload distribution. Among these, the tasks are prioritized as
13 high, 9 medium, and 5 low priority. By automating task tracking and reminders, project
coordination is enhanced as team members are promptly informed of overdue responsibilities
and upcoming deadlines, allowing for timely interventions and resource reallocation. This
automation aids in effective deadline management by ensuring that high-priority tasks
receive the attention they need, thus minimizing project delays and promoting more
efficient workflow management. Overall, it supports teams in maintaining focus on critical
tasks, optimizing their productivity and ensuring project milestones are met.


###Problem 5: Concept Your Own Problem

A business problem I am interested in is automating a content upload queue. At work, publishing information can be spread across different spreadsheets and tracking systems, so pulling together one clean queue of items ready for upload can take a lot of manual review. This creates extra work and makes it easier for something to be missed.

If I built this further in Python, I would read the publishing data from spreadsheets, apply rules to identify which items are ready for upload, skip incomplete records, and combine the results into one organized queue. The final output could be a report or spreadsheet tab showing the items ready for upload and the key details the team needs to move them forward. This would save time, improve consistency, and make the publishing workflow easier to manage.

I think this is strong because it matches the assignment’s “problem of your own” requirement while staying grounded in a real business workflow.